# ATML PA2 - Colab launcher

Thin launcher only: clone/pull, install, mount Drive, link the big folders to Drive, get the course assets, then call the scripts. All experiment logic lives in the `.py` files.

Set the runtime to a GPU first (Runtime > Change runtime type).

**Every new session:** run sections 1-5 again before anything else. A fresh runtime has no GPU selected, no data files and no Drive links, and the scripts stop with a message saying which of these is missing.

### 1. GPU check

In [ ]:
!nvidia-smi

### 2. Clone or update the repo

In [ ]:
%cd /content
!test -d atml_PA2 || git clone https://github.com/Zeeba1705/atml_PA2.git
%cd /content/atml_PA2
!git fetch origin
!git checkout task2-ppo
!git pull
!git log --oneline -3

### 3. Install the pinned requirements

In [ ]:
!pip install -q -r requirements.txt

### 4. Mount Drive and link the large folders

`checkpoints/ cached/ outputs/ results/` live on Drive so they survive a dead session.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
DRIVE=/content/drive/MyDrive/atml_pa2

for d in checkpoints cached outputs results; do
    mkdir -p "$DRIVE/$d"

    if [ ! -L "$d" ]; then
        if [ -d "$d" ]; then
            cp -rn "$d/." "$DRIVE/$d/"
            rm -rf "$d"
        fi

        ln -s "$DRIVE/$d" "$d"
    fi
done

ls -l

### 5. Course assets

Downloads only when validation fails.

In [ ]:
!python -m scripts.validate_assets || (python -m scripts.download_assets && python -m scripts.validate_assets)

### 6. Environment check and unit tests

In [ ]:
!python -m scripts.check_environment
!python -m pytest tests -q

### 7. Task 1 - quick test

32 examples of the real model, a checkpoint after every optimizer step. Run this before the full run.

In [ ]:
!python -m task1_dpo.train --config configs/dpo.yaml --run-name quicktest --max-examples 32 --save-every 1 --resume

In [ ]:
!ls -R outputs/task1_dpo/quicktest | head -30
!cat results/task1_dpo/quicktest/train_log.jsonl
!cat results/task1_dpo/quicktest/train_metrics.json

### 8. Task 1 - standard DPO (one epoch)

If the session dies, rerun cells 2-5 and then this cell again: `--resume` continues from the latest checkpoint.

In [ ]:
!python -m task1_dpo.train --config configs/dpo.yaml --run-name standard --resume

In [ ]:
!tail -n 5 results/task1_dpo/standard/train_log.jsonl
!cat results/task1_dpo/standard/train_metrics.json

### 9. Task 1 - evaluation quick test

8 held-out pairs and one sample per word-limit prompt, on the quick-test adapter. Run this before the full evaluations.

In [ ]:
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/quicktest --name quicktest --max-examples 8 --word-limit-samples 1

In [ ]:
!cat results/task1_dpo/quicktest/eval_metrics.json
!head -n 2 results/task1_dpo/quicktest/generations.jsonl

### 10. Task 1 - evaluate the standard run and the untrained reference

Same held-out pairs, seed and decoding for both. No `--adapter` means the reference policy.

In [ ]:
!python -m task1_dpo.evaluate --config configs/dpo.yaml --adapter outputs/task1_dpo/standard --name standard

In [ ]:
!python -m task1_dpo.evaluate --config configs/dpo.yaml --name reference

In [ ]:
!cat results/task1_dpo/standard/eval_metrics.json
!cat results/task1_dpo/reference/eval_metrics.json

### 11. Task 1 - beta study

Trains and evaluates the three short forks. Runs that are already finished are skipped, so this cell is safe to rerun.

In [ ]:
!python -m task1_dpo.ablate_beta --config configs/dpo.yaml

In [ ]:
!cat results/task1_dpo/beta_ablation.json

### 12. Task 1 - length study

Needs the standard adapter. Trains the length-balanced model, then evaluates reference, standard and length-balanced on the length-stratified pairs and the word-limit prompts.

In [ ]:
!python -m task1_dpo.analyze_length --config configs/dpo.yaml

In [ ]:
!cat results/task1_dpo/length_analysis.json
!cat results/task1_dpo/filtered_examples.json | head -n 40

### 13. Task 1 - rerun the beta forks with step retry (one time)

The first beta forks skipped optimizer steps with non-finite gradients (0, 1 and 3 of 37), so they did not get equal updates. This moves them to `_superseded/` and reruns the study with the retry code. Run the move cell once only: it stops with an error if the forks were already moved.

In [ ]:
%%bash
set -e
mkdir -p results/task1_dpo/_superseded outputs/task1_dpo/_superseded

for b in 0.03 0.1 0.3; do
    mv results/task1_dpo/beta_$b results/task1_dpo/_superseded/
    mv outputs/task1_dpo/beta_$b outputs/task1_dpo/_superseded/
done

mv results/task1_dpo/beta_ablation.json results/task1_dpo/_superseded/

echo "beta forks trained before step retry: optimizer steps with non-finite gradients were skipped (beta_0.03: 0, beta_0.1: 1, beta_0.3: 3 of 37). Superseded by the rerun." > results/task1_dpo/_superseded/NOTE.txt

ls results/task1_dpo results/task1_dpo/_superseded outputs/task1_dpo

In [ ]:
!python -m task1_dpo.ablate_beta --config configs/dpo.yaml

In [ ]:
!cat results/task1_dpo/beta_ablation.json
!grep -h '"run_name"\|"step_retries_total"\|"steps_with_retries"\|"skipped_optimizer_steps"\|"git_commit"' results/task1_dpo/beta_*/train_metrics.json

### 14. Task 2 - quick test

2 updates of the real models from the supplied midpoint, under the run name `quicktest`. Run this before the standard run.

In [ ]:
!python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name quicktest --updates 2 --resume

In [ ]:
!ls -R outputs/task2_ppo/quicktest | head -30
!cat results/task2_ppo/quicktest/train_log.jsonl
!cat results/task2_ppo/quicktest/train_metrics.json

### 15. Task 2 - standard PPO continuation (20 updates)

If the session dies, rerun cells 2-5 and then this cell again: `--resume` continues from the latest checkpoint.

In [ ]:
!python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard --resume

In [ ]:
!tail -n 3 results/task2_ppo/standard/train_log.jsonl
!cat results/task2_ppo/standard/train_metrics.json

### 16. Task 2 - evaluation quick test

8 held-out prompts on the quick-test adapter. Run this before the full evaluations.

In [ ]:
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/quicktest --name quicktest --max-examples 8

In [ ]:
!cat results/task2_ppo/quicktest/eval_metrics.json
!head -n 2 results/task2_ppo/quicktest/generations.jsonl

### 17. Task 2 - evaluate the base policy, the midpoint and the standard run

Same held-out prompts, seed, batch size and decoding for all three. No `--adapter` means the untouched base policy.

In [ ]:
!python -m task2_ppo.evaluate --config configs/ppo.yaml --name sft

In [ ]:
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter checkpoints/ppo_midpoint_policy --name midpoint

In [ ]:
!python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard

In [ ]:
!cat results/task2_ppo/sft/eval_metrics.json
!cat results/task2_ppo/midpoint/eval_metrics.json
!cat results/task2_ppo/standard/eval_metrics.json

### 18. Task 2 - clipping study, cached batch

No training. Scores the 32 cached rollouts under the midpoint policy, and under the standard adapter when it exists.

In [ ]:
!python -m task2_ppo.analyze_clipping --config configs/ppo.yaml --part cached

In [ ]:
!cat results/task2_ppo/clipping_cached_midpoint.json
!cat results/task2_ppo/clipping_cached_standard.json

### 19. Task 2 - clipping study, forks

Needs the finished standard run. Trains and evaluates `clip_0.05`, `clip_0.2` and `clip_0.5` (8 updates each from the midpoint). Forks that are already finished are skipped, so this cell is safe to rerun.

In [ ]:
!python -m task2_ppo.analyze_clipping --config configs/ppo.yaml --part forks

In [ ]:
!cat results/task2_ppo/stability_threshold.json
!cat results/task2_ppo/clipping_forks.json